# Base Server

> The core abstraction for different FL Servers. Any general Server functionality resides here. All other server must inherit from this class

In [1]:
#| default_exp servers.base_server

In [2]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [3]:
#| export
import gc
from fastcore.utils import patch

from tqdm import tqdm
from loguru import logger
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F

from fedai.client_selector import BaseClientSelector
from fedai.data import prepare_dl
from fedai.models import create_model
from fedai.optimizers import get_optimizer
from fedai.core import get_clean_kwargs

In [4]:
#| export
class StateManager:
    def __init__(self):
        self.registry = {} 

    def set_state(self, client_id, state_dict):
        """
        Takes a dictionary of tensors/objects and moves them to CPU.
        Works for weights, optimizer states, anchors (h), etc.
        """
        if client_id not in self.registry:
            self.registry[client_id] = {}

        for key, value in state_dict.items():
            # Deep clone and move to CPU if it's a torch object
            if isinstance(value, torch.Tensor):
                self.registry[client_id][key] = value.detach().cpu().clone()
            elif isinstance(value, dict):
                # Handle nested dicts (like optimizer.state_dict())
                self.registry[client_id][key] = self._to_cpu(value)
            else: # handles non-torch objects (like lists, scalars, etc.)
                self.registry[client_id][key] = value

    def get_state(self, client_id):       
        if client_id not in self.registered_ids:
            init_state = {}
            return init_state
     
        return copy.deepcopy(self.registry.get(client_id, {}))

    def _to_cpu(self, obj):
        """Recursively moves tensors in a nested structure to CPU."""
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().clone()
        elif isinstance(obj, dict):
            return {k: self._to_cpu(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [self._to_cpu(v) for v in obj]
        return obj

    @property
    def registered_ids(self):
        return list(self.registry.keys())

In [5]:
#| hide
from fedai.models import create_model
from fedai.optimizers import get_optimizer
from omegaconf import OmegaConf
from fedai.client_selector import BaseClientSelector
import importlib
from fedai.cfgs.data import *
from fedai.cfgs.models import *
from fedai.cfgs.algos import *
from fedai.cfgs.optimizers import *
from fedai.core import get_clean_kwargs
from fedai.cfgs import MainConfig

In [8]:
#| hide
cfg = MainConfig()
cfg.model = LeNetConfig()
cfg.data = MNISTConfig()
cfg.algorithm = pFedMeConfig()
cfg.optimizer = pFedMeOptimizerConfig()
model = create_model(cfg)
kwargs = get_clean_kwargs(cfg.optimizer)
kwargs.pop('cls', None)

opt = get_optimizer(cfg)(model.parameters(), **kwargs)

TypeError: pFedMeOptimizer.__init__() got an unexpected keyword argument 'lambda_'

In [7]:
opt

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

In [ ]:
#| hide
st_mgr = StateManager()
st_mgr.set_state(1, {'model': model.state_dict(), 'optimizer': opt.state_dict()})

In [ ]:
#| hide
st_mgr.get_state(1)

{'model': {'backbone.conv1.0.weight': tensor([[[[ 8.4821e-02,  9.2751e-02,  2.8823e-02, -8.1678e-02, -1.1420e-01],
            [ 9.3485e-02,  9.7398e-02, -5.7113e-02, -1.5102e-02, -7.8586e-02],
            [ 8.4367e-03,  6.6497e-02, -7.8845e-02,  3.2614e-02,  7.7886e-02],
            [-2.0501e-02, -2.8694e-02, -1.1972e-02, -5.9721e-02,  2.7669e-02],
            [ 8.4279e-02, -3.0029e-02,  1.2282e-02, -3.0839e-02,  1.8090e-02]],
  
           [[-5.9385e-02, -4.5338e-02, -6.3399e-02, -7.5960e-02, -9.4236e-02],
            [ 9.5013e-03,  9.3175e-02,  1.1315e-02,  9.9914e-03,  6.1940e-02],
            [ 4.2970e-02,  1.8318e-02, -1.1351e-02, -1.3393e-03, -6.7036e-02],
            [-3.6667e-02, -2.7931e-02, -5.9913e-04, -1.1361e-01,  8.2081e-02],
            [-7.8382e-03, -3.1292e-02,  5.6871e-02, -3.0061e-02, -8.1601e-03]],
  
           [[ 7.5030e-02, -8.1843e-02, -5.7203e-02,  5.6347e-02,  6.0826e-02],
            [-9.6175e-02, -5.4130e-02, -4.8834e-02,  9.6384e-02,  2.2792e-02],
        

In [ ]:
#| hide
opt.load_state_dict(st_mgr.get_state(1)['optimizer'])

In [ ]:
#| hide
st_mgr.get_state(2)

{}

## BaseServer

This class implements basic components of the Server abstraction and is considered the base class for all type of Servers

In [ ]:
#| export
class BaseServer:
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        self.cfg = cfg 
        self.client_selector = client_selector
        self.client_cls = client_cls
        self.criterion = criterion
        self.writer = writer
        self.fds = fds
        self.device = device
        self.model = create_model(self.cfg)
        for key, value in kwargs.items():
            setattr(self, key, value)
        self.state_mgr = StateManager()
        self.logger = logger

We need to make sure that all clients start from the same initial model at round 0. So, we need to save the initial model once at the server side and load it at each client during its initialization.

In [ ]:
#| hide
from fedai.data import init_data


In [ ]:
#| hide
fds = init_data(cfg.data)

In [ ]:
#| hide
client_cls = importlib.import_module(f"fedai.clients.base_client")
client_cls= getattr(client_cls, "BaseClient")
client_selector = BaseClientSelector(cfg)
server = BaseServer(cfg= cfg,
                    client_selector= client_selector,
                    client_cls= client_cls,
                    criterion= nn.CrossEntropyLoss(),
                    fds= fds,
                    kwargs= {"anything": "value"}
                    )


In [ ]:
#| export
@patch
def client_fn(self: BaseServer, id, comm_round, client_state):

    if (comm_round == 1 and client_state == {}) or client_state == {}:
        client_state['model'] = self.model.state_dict()

    model = create_model(self.cfg)
    model.load_state_dict(client_state['model'])
    client_state['model'] = model
    
    kwargs = get_clean_kwargs(self.cfg.optimizer)
    kwargs.pop("cls", None)
    optimizer = get_optimizer(self.cfg)(model.parameters(), **kwargs)
    optimizer.load_state_dict(client_state['optimizer']) if 'optimizer' in client_state else None
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)

    client_state['optimizer'] = optimizer

    train_loader = prepare_dl(self.cfg.data, id, self.fds, train=True, distributed=False)
    test_loader = prepare_dl(self.cfg.data, id, self.fds, train=False, distributed=False)
    client = self.client_cls(id= id, 
                             cfg= self.cfg,
                             train_loader= train_loader,
                             test_loader= test_loader,
                             state= client_state,
                             criterion= self.criterion,
                             device= self.device,
                             t= comm_round)
    client.logger = self.logger
    return client


In [ ]:
#| hide
client = server.client_fn(id= 0,
                          comm_round= 1,
                          client_state= st_mgr.get_state(0),
                          )
client

<fedai.clients.base_client.BaseClient>

In [ ]:
#| export
@patch
def train(self: BaseServer):
    res =  []
    selected_clients = self.client_selector.select()
    
    for t in range(1, self.cfg.n_rounds + 1):
        lst_active_ids = selected_clients[t-1]
        len_clients_ds = {}

        for id in lst_active_ids:
            client_state = self.state_mgr.get_state(id)
            client = self.client_fn(id= id, comm_round= t, client_state= client_state)
            client.fit()
            self.logger.info(f"Client {id} finished training.")
            self.logger.info("*"*20)
            self.state_mgr.set_state(id, client.save_state())
            
            len_clients_ds[id] = len(client.train_loader.dataset)

            del client 
            gc.collect()
            torch.cuda.empty_cache()

        self.aggregate(lst_active_ids, t, len_clients_ds)#lst_active_ids, comm_round, len_clients_ds
        train_res, test_res = self.evaluate(t)
        # self.state_mgr.registry.clear() if self.cfg.agg == "one_model" else None
        
        train_df, test_df = self.writer.write(lst_active_ids, train_res, test_res, t) 
        res.append((train_df, test_df))
        
    self.writer.save(res)
    self.writer.finish()

    return res

Each subclass, should implement the `aggregate` method, which is responsible for aggregating the local models from the clients and updating the global model/local personalized models accordingly.

In [ ]:
#| export
@patch
def aggregate(self: BaseServer):
    raise NotImplementedError("This method should be implemented by subclasses.")
        

### Evaluation




All methods are evaluated using the average of the local evaluation results across all clients.

In [ ]:
#| export
@patch
def evaluate(self: BaseServer, t):
    lst_train_res = []
    lst_test_res = []
    for id in range(self.cfg.num_clients):
        client_state = self.state_mgr.get_state(id)
        client = self.client_fn(id= id, comm_round= t, client_state= client_state)
        
        res_train = client.evaluate_local(loader= 'train')
        lst_train_res.append(res_train)

        res_test = client.evaluate_local(loader= 'test')
        lst_test_res.append(res_test)
        
        del client
        gc.collect()
        torch.cuda.empty_cache()

    return lst_train_res, lst_test_res    


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()